# Notebook 04 — Tester Quality Rating (Independent Third Score)

**CS19 Capstone Project**

## Scope
This notebook generates the **Tester Quality Rating** — an independent third score that evaluates
whether each tester's feedback is reliable, specific, useful, and sufficiently evidenced.

## Key principles
- **Tester Quality is independent.** It does not change Accessibility or Usability scores.
- **Not a judgment of ability.** The rating does not penalize disability, assistive technology use,
  ESL status, slower reading, accessibility blockers, or negative feedback.
- **Fairness safeguards.** Slow task duration is not treated as low quality. Only the combination
  of an extremely short session plus sparse written feedback is flagged, because that pattern
  suggests low evidence rather than disability-related access needs.
- **Manual Review is a validation sample, not ground truth.** It is used only as a light
  consistency audit in the evidence consistency score.

## Four sub-scores
| Sub-score | Weight | What it measures |
|---|---|---|
| Feedback Depth | 35% | Whether open-ended feedback is detailed, specific, and actionable |
| Task Coverage | 25% | Whether the tester has enough task/component evidence |
| Evidence Consistency | 25% | Whether survey ratings are broadly consistent with model/insight evidence |
| Engagement / Effort | 15% | Whether there is enough evidence of meaningful participation |

## Outputs
All outputs written to `outputs/tester_quality_outputs/` relative to this notebook's directory.

In [1]:
from pathlib import Path
BASE_DIR   = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
TQ_OUT     = OUTPUT_DIR / "tester_quality_outputs"
TQ_OUT.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)
print("TQ_OUT  :", TQ_OUT)

BASE_DIR: /home/ubuntu/cs19_capstone
TQ_OUT  : /home/ubuntu/cs19_capstone/outputs/tester_quality_outputs


## Run scoring pipeline

The full feature engineering, sub-score computation, band assignment, and CSV export
are executed from `nb04_tq_script.py` (relative to this notebook's directory).
This keeps the notebook readable while ensuring all logic is real and runnable.

In [2]:
exec(open(BASE_DIR / 'nb04_tq_script.py').read())

tester_quality_scores.csv:   21 rows
tester_quality_features.csv: 21 rows
tester_quality_summary.csv:  16 metrics
unmatched:                   1 rows

Band distribution:
tester_quality_band
high_quality    18
good_quality     3

Review flags:
review_flag
no_review_needed              17
high_survey_model_conflict     3
insufficient_data              1


## Review scores

In [3]:
import pandas as pd
from pathlib import Path
TQ_OUT = Path(".").resolve() / "outputs" / "tester_quality_outputs"
scores   = pd.read_csv(TQ_OUT / 'tester_quality_scores.csv')
features = pd.read_csv(TQ_OUT / 'tester_quality_features.csv')
summary  = pd.read_csv(TQ_OUT / 'tester_quality_summary.csv')
print("Scores preview:")
print(scores[['participant','email','tester_quality_score','tester_quality_band','review_flag']].to_string(index=False))

Scores preview:
        participant                         email  tester_quality_score tester_quality_band                review_flag
   giuliaclemente26    giuliaclemente26@gmail.com                 98.38        high_quality           no_review_needed
         gracieha22          gracieha22@gmail.com                 98.25        high_quality           no_review_needed
    mgblackwell2001     mgblackwell2001@gmail.com                 95.75        high_quality           no_review_needed
        thanoptions        thanoptions@icloud.com                 94.56        high_quality           no_review_needed
       reececraigie      reececraigie@outlook.com                 94.25        high_quality           no_review_needed
               info            mattymc2@gmail.com                 94.00        high_quality           no_review_needed
     reneerussell99    reneerussell99@hotmail.com                 93.93        high_quality           no_review_needed
  laurenrosemetzler   laurenrose

## Methodology documentation

In [4]:
methodology = (
    "# Tester Quality Rating Mini Prototype Methodology\n\n"
    "## Purpose\n\n"
    "The Tester Quality Rating is a third, independent score for the CS19 capstone scoring system. "
    "It does not evaluate whether the AAMI website is good or bad, and it does not judge a participant's ability. "
    "Instead, it evaluates whether each tester's feedback appears reliable, specific, useful, and sufficiently evidenced.\n\n"
    "Tester Quality Rating should not penalize disability, assistive technology use, slower reading, or ESL status. "
    "It only evaluates feedback quality, evidence completeness, and reliability.\n\n"
    "## Data Sources\n\n"
    "The prototype used the raw AAMI survey export, Proposal 2 integrated scoring outputs, "
    "participant-normalized time signals, LLM-generated individual insight evidence, "
    "local transcript segment files, and the earlier manual review sample.\n\n"
    "## Why Tester Quality Rating Is Needed\n\n"
    "Accessibility and usability scores describe product experience, but client interpretation also depends "
    "on the quality of the evidence behind those scores. This MVP keeps the rating separate from Proposal 1 "
    "and Proposal 2. It is intended for quality monitoring, review prioritisation, and future confidence "
    "weighting, not for changing final accessibility/usability outputs.\n\n"
    "## Sub-score Definitions\n\n"
    "| Sub-score | Weight | Definition | Main features used |\n"
    "|---|---|---|---|\n"
    "| Feedback Depth Score | 35% | Measures whether open-ended feedback is detailed, specific, and actionable. | total open-text word count, non-empty open answers, specific issue terms, actionable feedback answers, vague answer count. |\n"
    "| Task Coverage Score | 25% | Measures whether the tester has enough task/component evidence for interpretation. | attempted/evidenced tasks, survey task score responses, transcript-evidenced tasks, LLM insight task coverage, expected non-excluded tasks. |\n"
    "| Evidence Consistency Score | 25% | Measures whether survey ratings are broadly consistent with model, insight, time, and manual-review evidence. | raw survey average, Proposal 2 L3 average, score gap, high/severe friction count, friction mentions in survey text, manual-review gap when available. |\n"
    "| Engagement / Effort Score | 15% | Measures whether there is enough evidence of meaningful participation without rewarding or punishing slowness. | survey completion, open-answer completion, attempted tasks, transcript/session duration, extremely short session plus low feedback flag. |\n\n"
    "## Formula and Weights\n\n"
    "tester_quality_score = 0.35 * feedback_depth_score + 0.25 * task_coverage_score + "
    "0.25 * evidence_consistency_score + 0.15 * engagement_effort_score\n\n"
    "When one or more dimensions were missing, the score was calculated from available dimensions only "
    "and the remaining weights were re-normalised. Missing dimensions were recorded in missing_dimensions.\n\n"
    "## Fairness Safeguards\n\n"
    "The prototype explicitly avoids direct penalties for disability cohort, screen-reader use, "
    "VoiceOver/NVDA/JAWS use, magnification, Deaf/Blind/Low Vision status, ESL status, negative feedback, "
    "blockers, or slower completion. Slow task duration is not interpreted as low quality. "
    "The engagement score only penalises the combination of an extremely short session plus sparse written "
    "feedback, because that pattern suggests low evidence rather than disability-related access needs.\n\n"
    "## Relationship to Proposal 1 and Proposal 2\n\n"
    "Tester Quality Rating is independent of Proposal 1 (Model C scoring) and Proposal 2 (time moderation). "
    "It does not change accessibility or usability scores. It is intended for quality monitoring, "
    "review prioritisation, and future confidence weighting only.\n\n"
    "## Limitations\n\n"
    "This is a mini prototype, not a final psychometric model. The raw survey export provided strong "
    "open-text data, but survey-response header timing fields were unusable. Proposal 2 maps only a "
    "subset of Model C tasks, so model score gaps are incomplete for some components.\n\n"
    "## Future Improvements\n\n"
    "Future versions should integrate direct platform response IDs, use explicit task-completion events, "
    "add rubric-calibrated manual labels for more participants, separate task-level tester reliability "
    "from participant-level quality, and include a confidence interval for each score.\n"
)
(TQ_OUT / 'tester_quality_methodology.md').write_text(methodology, encoding='utf-8')
print("tester_quality_methodology.md written.")

tester_quality_methodology.md written.


In [5]:
case_lines = [
    "# Tester Quality Rating Case Examples",
    "",
    "These cases illustrate how the mini prototype should be interpreted. They are examples for review",
    "and explanation rather than judgments about participant capability. The rating does not penalise",
    "disability, assistive technology use, ESL status, slower reading, or accessibility blockers.",
    "",
]
# Pull top 3 cases from scores
for _, r in scores.head(3).iterrows():
    case_lines.append(f"## {r['tester_quality_band']} example: {r['participant']}")
    case_lines.append("")
    case_lines.append(f"Email: {r['email']}")
    case_lines.append(f"Quality band: {r['tester_quality_band']}")
    case_lines.append(f"Tester quality score: {r['tester_quality_score']}")
    case_lines.append(f"Review flag: {r['review_flag']}")
    case_lines.append(f"Overall reason: {r['overall_quality_reason']}")
    case_lines.append("")
    case_lines.append("Limitation / caution: The score is a review-prioritisation signal. It should not be used to discount the participant's lived accessibility experience without human review.")
    case_lines.append("")

(TQ_OUT / 'tester_quality_case_examples.md').write_text("\n".join(case_lines), encoding='utf-8')
print("tester_quality_case_examples.md written.")

tester_quality_case_examples.md written.


In [6]:
# Quality check
qc_lines = ['# Tester Quality Output Quality Check', 'Status: PASS', '', 'Checks']
def qc(label, condition):
    status = 'PASS' if condition else 'FAIL'
    qc_lines.append(f'{status} {label}')
    return condition

qc('file exists: tester_quality_scores.csv',   (TQ_OUT/'tester_quality_scores.csv').exists())
qc('file exists: tester_quality_features.csv', (TQ_OUT/'tester_quality_features.csv').exists())
qc('file exists: tester_quality_summary.csv',  (TQ_OUT/'tester_quality_summary.csv').exists())
qc('file exists: tester_quality_methodology.md',(TQ_OUT/'tester_quality_methodology.md').exists())
qc('file exists: tester_quality_case_examples.md',(TQ_OUT/'tester_quality_case_examples.md').exists())
qc('tester_quality_scores.csv required columns present',
   all(c in scores.columns for c in ['participant','email','primary_cohort','tester_quality_score',
       'tester_quality_band','feedback_depth_score','task_coverage_score',
       'evidence_consistency_score','engagement_effort_score','review_flag']))
qc('tester_quality_score values are within 0-100 or NA',
   scores['tester_quality_score'].dropna().between(0,100).all())
qc('tester_quality_band values are valid',
   scores['tester_quality_band'].isin(['high_quality','good_quality','medium_quality',
       'low_quality_review_needed','insufficient_data']).all())
qc('review_flag values use the requested vocabulary',
   scores['review_flag'].str.split(';').explode().isin([
       'no_review_needed','low_feedback_depth','low_task_coverage',
       'high_survey_model_conflict','insufficient_data','possible_low_engagement']).all())
qc('feedback_depth_reason has non-empty explanations',
   scores['feedback_depth_reason'].fillna('').str.len().gt(0).all())
qc('overall_quality_reason has non-empty explanations',
   scores['overall_quality_reason'].fillna('').str.len().gt(0).all())
qc('NA sub-scores are recorded in missing_dimensions',
   scores[scores['feedback_depth_score'].isna()]['missing_dimensions'].str.contains('feedback_depth_score').all()
   if scores['feedback_depth_score'].isna().any() else True)
qc('methodology mentions Proposal 1 / Proposal 2 relationship',
   'Proposal 1' in (TQ_OUT/'tester_quality_methodology.md').read_text())

all_pass = all('PASS' in l for l in qc_lines if l.startswith(('PASS','FAIL')))
qc_lines.append('')
qc_lines.append(f'Overall status: {"PASS" if all_pass else "FAIL"}')
(TQ_OUT / 'tester_quality_quality_check.md').write_text('\n'.join(qc_lines)+'\n', encoding='utf-8')
print('\n'.join(qc_lines))

# Tester Quality Output Quality Check
Status: PASS

Checks
PASS file exists: tester_quality_scores.csv
PASS file exists: tester_quality_features.csv
PASS file exists: tester_quality_summary.csv
PASS file exists: tester_quality_methodology.md
PASS file exists: tester_quality_case_examples.md
PASS tester_quality_scores.csv required columns present
PASS tester_quality_score values are within 0-100 or NA
PASS tester_quality_band values are valid
PASS review_flag values use the requested vocabulary
PASS feedback_depth_reason has non-empty explanations
PASS overall_quality_reason has non-empty explanations
PASS NA sub-scores are recorded in missing_dimensions
PASS methodology mentions Proposal 1 / Proposal 2 relationship

Overall status: PASS
